# Conditional moments, probabilities, and CDF uncertainty

All queries apply the learned operator directly. The CDF is projected only after the full signed curve has been evaluated. Execute a partir da raiz `code/fsnm`. O notebook grava figuras apenas em `code/fsnm/figures`; a cópia para o diretório TeX é deliberadamente manual.

## Fit once and reuse the empirical response marginal

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from fsnm import empirical_loss, fit_fsnm


def basis_matrix(values, rank=3):
    values = np.asarray(values)
    basis = np.column_stack(
        [
            np.sqrt(2) * np.sin(np.pi * values),
            np.sqrt(2) * np.cos(np.pi * values),
            np.sqrt(2) * np.sin(2 * np.pi * values),
        ]
    )
    return basis[:, :rank]


def kappa_exact(x_values, y_values):
    return 1 + (basis_matrix(x_values) * SIGMAS) @ basis_matrix(y_values).T


def sample_joint(size, seed):
    rng = np.random.default_rng(seed)
    upper_bound = 1 + 2 * SIGMAS.sum()
    x_parts = []
    y_parts = []
    n_accepted = 0

    while n_accepted < size:
        x = rng.uniform(-1, 1, size)
        y = rng.uniform(-1, 1, size)
        density_ratio = 1 + np.sum(
            basis_matrix(x) * SIGMAS * basis_matrix(y), axis=1
        )
        accepted = rng.uniform(size=size) < density_ratio / upper_bound
        x_parts.append(x[accepted])
        y_parts.append(y[accepted])
        n_accepted += accepted.sum()

    return np.concatenate(x_parts)[:size], np.concatenate(y_parts)[:size]

from sklearn.isotonic import IsotonicRegression

SIGMAS = np.array([0.18, 0.16, 0.12])
RANK = 3
N_TRAIN = 10_000
X_EVALUATION = np.array([-0.6, 0.0, 0.6])
CDF_GRID = np.linspace(-1, 1, 301)
TREE_PARAMETERS = dict(
    rank=RANK, n_iterations=11, step_size=0.1,
    max_depth=3, min_samples_leaf=300, seed=0,
)

x_train, y_train = sample_joint(N_TRAIN, seed=12)
phi, psi, singular_values, _ = fit_fsnm(x_train, y_train, **TREE_PARAMETERS)
grid = np.linspace(-1, 1, 160)
phi_grid = phi.predict(grid[:, None])


## Mean, variance, and exceedance probability

In [2]:

integration_grid = np.linspace(-1, 1, 4_001)
exact_kernel = kappa_exact(grid, integration_grid)
estimated_kernel = 1 + (phi_grid * singular_values) @ psi.predict(y_train[:, None]).T

def exact_query(g):
    return 0.5 * np.trapezoid(exact_kernel * g[None, :], integration_grid, axis=1)

def estimated_query(g):
    # Direct empirical operator: no clipping or normalization of kernel weights.
    return np.mean(estimated_kernel * g[None, :], axis=1)

exact_mean = exact_query(integration_grid)
estimated_mean = estimated_query(y_train)
exact_second = exact_query(integration_grid**2)
estimated_second = estimated_query(y_train**2)
exact_variance = exact_second - exact_mean**2
estimated_variance = estimated_second - estimated_mean**2
exact_tail = exact_query((integration_grid > 0.5).astype(float))
estimated_tail = estimated_query((y_train > 0.5).astype(float))

queries = [
    (r"$\mathbb{E}[Y\mid X=x]$", exact_mean, estimated_mean),
    (r"$\mathrm{Var}(Y\mid X=x)$", exact_variance, estimated_variance),
    (r"$\mathbb{P}(Y>0.5\mid X=x)$", exact_tail, estimated_tail),
]

figure, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
for axis, (title, exact, estimated) in zip(axes, queries):
    axis.plot(grid, exact, color="black", linewidth=2, label="Exact")
    axis.plot(grid, estimated, color="tab:red", linewidth=2, linestyle="--", label="FSNM")
    axis.set(title=title, xlabel="$x$")
    print(f"{title} RMSE: {np.sqrt(np.mean((estimated - exact) ** 2)):.4f}")
axes[0].set_ylabel("Conditional functional")
axes[0].legend(frameon=False)
figure_directory = Path.cwd() / "figures"
figure_directory.mkdir(exist_ok=True)
query_path = figure_directory / "01_conditional_queries.png"
figure.savefig(query_path, dpi=200, bbox_inches="tight")
print("saved:", query_path)


$\mathbb{E}[Y\mid X=x]$ RMSE: 0.0237
$\mathrm{Var}(Y\mid X=x)$ RMSE: 0.0166
$\mathbb{P}(Y>0.5\mid X=x)$ RMSE: 0.0233


saved: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/01_conditional_queries.png


## Direct indicator queries and bootstrap CDF bands

In [3]:

def exact_conditional_cdf():
    density = kappa_exact(X_EVALUATION, CDF_GRID)
    increments = 0.25 * (density[:, :-1] + density[:, 1:]) * np.diff(CDF_GRID)
    return np.column_stack([np.zeros(len(X_EVALUATION)), np.cumsum(increments, axis=1)])

def project_rows_to_cdf(raw_cdf):
    """Project each complete signed curve, not its individual kernel weights."""
    projected = np.empty_like(raw_cdf)
    locations = np.arange(raw_cdf.shape[1] + 2)
    weights = np.ones(raw_cdf.shape[1] + 2)
    weights[[0, -1]] = 1e6
    projector = IsotonicRegression(y_min=0.0, y_max=1.0)
    for row, values in enumerate(raw_cdf):
        anchored = np.r_[0.0, values, 1.0]
        projected[row] = projector.fit_transform(
            locations, anchored, sample_weight=weights,
        )[1:-1]
    return projected

def estimated_conditional_cdf(model, y_marginal):
    phi_model, psi_model, values = model
    weights = 1 + (
        phi_model.predict(X_EVALUATION[:, None]) * values
    ) @ psi_model.predict(y_marginal[:, None]).T
    order = np.argsort(y_marginal)
    sorted_y = y_marginal[order]
    direct_cumulative = np.cumsum(weights[:, order], axis=1) / len(y_marginal)
    indices = np.searchsorted(sorted_y, CDF_GRID, side="right") - 1
    raw_cdf = np.zeros((len(X_EVALUATION), len(CDF_GRID)))
    available = indices >= 0
    raw_cdf[:, available] = direct_cumulative[:, indices[available]]
    return project_rows_to_cdf(raw_cdf)

def bootstrap_cdfs(x, y, n_bootstrap=100):
    rng = np.random.default_rng(2026)
    replicates = np.empty((n_bootstrap, len(X_EVALUATION), len(CDF_GRID)))
    for bootstrap in range(n_bootstrap):
        indices = rng.integers(0, len(y), size=len(y))
        bootstrap_model = fit_fsnm(x[indices], y[indices], **TREE_PARAMETERS)[:3]
        replicates[bootstrap] = estimated_conditional_cdf(
            bootstrap_model, y[indices],
        )
        if (bootstrap + 1) % 10 == 0:
            print(f"bootstrap {bootstrap + 1}/{n_bootstrap}", flush=True)
    return replicates

exact_cdf = exact_conditional_cdf()
estimate_cdf = estimated_conditional_cdf((phi, psi, singular_values), y_train)
bootstrap = bootstrap_cdfs(x_train, y_train, n_bootstrap=100)
lower, upper = np.percentile(bootstrap, [2.5, 97.5], axis=0)

coverage = np.mean((lower <= exact_cdf) & (exact_cdf <= upper), axis=1)
width = np.mean(upper - lower, axis=1)
for x_value, value_coverage, value_width in zip(X_EVALUATION, coverage, width):
    print(f"x={x_value:+.1f}: coverage={value_coverage:.3f}, width={value_width:.4f}")
print(f"average band width: {width.mean():.4f}")

figure, axes = plt.subplots(
    1, 3, figsize=(11.5, 3.2), sharex=True, sharey=True, constrained_layout=True,
)
for axis, x_value, exact, estimated, lo, hi in zip(
    axes, X_EVALUATION, exact_cdf, estimate_cdf, lower, upper,
):
    axis.fill_between(CDF_GRID, lo, hi, color="tab:blue", alpha=0.22, label="Pointwise 95% band")
    axis.plot(CDF_GRID, exact, color="black", linewidth=2, label="Exact")
    axis.plot(CDF_GRID, estimated, color="tab:blue", linewidth=1.8, linestyle="--", label="Estimate")
    axis.set(xlim=(-1, 1), ylim=(-0.02, 1.02), title=rf"$x={x_value:.1f}$", xlabel="$y$")
axes[0].set_ylabel("Conditional CDF")
axes[0].legend(frameon=False, loc="upper left")
cdf_path = figure_directory / "02_tree_conditional_cdf_uncertainty.png"
figure.savefig(cdf_path, dpi=200, bbox_inches="tight")
print("saved:", cdf_path)


bootstrap 10/100


bootstrap 20/100


bootstrap 30/100


bootstrap 40/100


bootstrap 50/100


...

bootstrap 80/100


bootstrap 90/100


bootstrap 100/100


x=-0.6: coverage=0.997, width=0.0664
x=+0.0: coverage=1.000, width=0.0648
x=+0.6: coverage=1.000, width=0.0653
average band width: 0.0655


saved: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/02_tree_conditional_cdf_uncertainty.png
